# Preprocessing

In [202]:
import pandas as pd

df_development = pd.read_csv("../datasets/development.csv", sep=",")
df_evaluation = pd.read_csv("../datasets/evaluation.csv", sep=",")

In [203]:
# Shape, label column, datatypes check for development
df_development.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79997 entries, 0 to 79996
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Id         79997 non-null  int64 
 1   source     79997 non-null  object
 2   title      79996 non-null  object
 3   article    79996 non-null  object
 4   page_rank  79997 non-null  int64 
 5   timestamp  79997 non-null  object
 6   label      79997 non-null  int64 
dtypes: int64(3), object(4)
memory usage: 4.3+ MB


In [204]:
# Shape, label column, datatypes check for evaluation
df_evaluation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Id         20000 non-null  int64 
 1   source     20000 non-null  object
 2   title      20000 non-null  object
 3   article    19999 non-null  object
 4   page_rank  20000 non-null  int64 
 5   timestamp  20000 non-null  object
dtypes: int64(2), object(4)
memory usage: 937.6+ KB


In [205]:
# Distribution Check for understand imbalanced or balanced?
df_development["label"].value_counts(normalize=True, sort=True)


label
0    0.294286
5    0.163169
2    0.139518
1    0.132355
3    0.124717
4    0.107179
6    0.038776
Name: proportion, dtype: float64

In [206]:
# Missing Value Check and Filling
print("Development data set\n", df_development.isnull().sum())

Development data set
 Id           0
source       0
title        1
article      1
page_rank    0
timestamp    0
label        0
dtype: int64


In [207]:
# Missing value filling
df_development.fillna("", inplace=True)

In [208]:

# Aggregation, all_text = source + title + article

df_development["all_text"] =df_development["source"] + " " + df_development["title"] + " " + df_development["article"]

# Feature Selection
X_train_val = df_development["all_text"] # source + title + article, I dropped the columns id, page_rank, timestamp
y_train_val = df_development["label"] # I separete the label (target)

In [209]:
# Cleaner function for TfidfVectorizer

import re

def clean_text(text):
    
    # The following text normalization steps were tested during preliminary experiments but were removed
    # because they resulted in lower Macro F1 scores

    #text = html.unescape(text) 
    # Converts HTML escap codes
    
    #text = re.sub(r"https?://\S+", " ", text) #
    # https?:// --> http:// or https://
    # \S+ --> rest of the url
    
    #text = re.sub(r"</?[^>]+>", " ", text)
    # Deletes anything between < and > or </ and >.
    # <..> text <..> but not delete the text
    
    
    text = re.sub(r"\\n|\\t|\n|\t|\\", " ", text)
    # Deletes the escape characters
    
    text = re.sub(r"\s+", " ", text).strip()
    # Deletes the extra spaces

    Apos_dict={"n't":" not","'m":" am","'ll":" will","'d":" would","'ve":" have","'re":" are"}
    # didn't add "'s":is --> because "'s" is not "is" every time
    
    for key,value in Apos_dict.items():
        if key in text:
            text=text.replace(key,value)

    # Fix the words with apostrophe
    # Example: we'll --> we will
    
    return text


# Model Selection + Hyperparameter Tuning

In [210]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import MultinomialNB

k = 5
skcv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
# I used Stratified since our data is imbalanced
# k = 3, 5, 8, 10 --> I tried them
# k = 3 --> decrease the F1_macro, k = 8 & 10 --> doesn't add any increase but make slower the model
# That's why I choosed k = 5 

scoring = "f1_macro"
# for GridSearchCV

labels = [0, 1, 2, 3, 4, 5, 6]
targets = {"label0", "label1", "label2","label3","label4","label5","label6"}
# labes and targets is for classification_report

Note:
-   For each model, the final code uses the best hyperparameters obtained from GridSearchCV.

-   I do not rerun the full hyperparameter search every time because it is computationally expensive and significantly increases execution time.

-   For reproducibility and transparency, I list all the tested hyperparameter ranges at the end of each model section.

-   I also provide the estimated runtime of each GridSearch on my machine to give an idea of the computational cost.

Pipeline:
-   TF-IDF vectorization → Classifier

Hyperparameters selected via GridSearchCV.

In [211]:
# LinearSVC

lsvc_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", sublinear_tf=True, preprocessor=clean_text)),
    ("clf_lsvc", LinearSVC(random_state=42, class_weight="balanced"))
])
lsvc_params = [{
    
    # Estimator
    
    #"clf_lsvc__random_state"=[42], 
    #"clf_lsvc__class_weight"=["balanced"]
    
    "clf_lsvc__max_iter":[1000],
    "clf_lsvc__C":[0.2],

    
    # TFIDF
    
    #"tfidf__stop_words":["english"],
    #"tfidf__sublinear_tf":[True],
    #"tfidf__preprocessor":[clean_text],
    
    "tfidf__max_features":[120000],
    "tfidf__max_df": [0.7],
    "tfidf__min_df": [3],
    "tfidf__analyzer": ["word"], 
    "tfidf__ngram_range": [(1, 2)]
}]

lsvc_gridcv = GridSearchCV(estimator=lsvc_pipe, param_grid=lsvc_params, scoring=scoring, cv=skcv,  n_jobs=-1, verbose=2, refit= True)

lsvc_gridcv.fit(X_train_val, y_train_val)

lsvc_best_params = lsvc_gridcv.best_params_

print(lsvc_gridcv.best_score_)
print(lsvc_best_params)

"""
Tried Parameters with GridSearchCV

# Estimator

"clf_lsvc__max_iter":[1000, 2000],
"clf_lsvc__C":[0.1, 0.2, 0.5, 1],
"clf_lsvc__class_weight":["balanced", None], 

# TFIDF

"tfidf__max_features":[50000, 80000, 100000, 120000, 160000],
"tfidf__max_df": [0.7, 0.75, 0.85, 0.9, 0.95],
"tfidf__min_df": [1, 2, 3],
"tfidf__sublinear_tf":[True, False],   

"tfidf__analyzer": ["word"], "tfidf__ngram_range": [(1,1), (1, 2), (1, 3)]
"tfidf__analyzer": ["char_wb"], "tfidf__ngram_range": [(3, 5), (4, 6)]

"""
# Estimated Run time: 2 minutes 40 seconds

Fitting 5 folds for each of 1 candidates, totalling 5 fits
0.7168774285019877
{'clf_lsvc__C': 0.2, 'clf_lsvc__max_iter': 1000, 'tfidf__analyzer': 'word', 'tfidf__max_df': 0.7, 'tfidf__max_features': 120000, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 2)}


'\nTried Parameters with GridSearchCV\n\n# Estimator\n\n"clf_lsvc__max_iter":[1000, 2000],\n"clf_lsvc__C":[0.1, 0.2, 0.5, 1],\n"clf_lsvc__class_weight":["balanced", None], \n\n# TFIDF\n\n"tfidf__max_features":[50000, 80000, 100000, 120000, 160000],\n"tfidf__max_df": [0.7, 0.75, 0.85, 0.9, 0.95],\n"tfidf__min_df": [1, 2, 3],\n"tfidf__sublinear_tf":[True, False],   \n\n"tfidf__analyzer": ["word"], "tfidf__ngram_range": [(1,1), (1, 2), (1, 3)]\n"tfidf__analyzer": ["char_wb"], "tfidf__ngram_range": [(3, 5), (4, 6)]\n\n'

In [212]:
# LinearSVC label base analysis

y_pred_lsvc = cross_val_predict(
    estimator = lsvc_gridcv.best_estimator_,
    X = X_train_val,
    y = y_train_val,
    cv=skcv
)

print(classification_report(y_train_val, y_pred_lsvc,labels=labels,target_names=targets))


              precision    recall  f1-score   support

      label6       0.76      0.75      0.76     23542
      label2       0.74      0.83      0.78     10588
      label3       0.81      0.85      0.83     11161
      label4       0.64      0.53      0.58      9977
      label1       0.79      0.95      0.86      8574
      label0       0.62      0.47      0.53     13053
      label5       0.57      0.84      0.68      3102

    accuracy                           0.73     79997
   macro avg       0.70      0.75      0.72     79997
weighted avg       0.72      0.73      0.72     79997



In [213]:
# Logistic Regression

lr_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", sublinear_tf=True, preprocessor=clean_text)),
    ("clf_lr", LogisticRegression(random_state=42, class_weight="balanced"))
])
lr_params =[{

    # Estimator

    #"clf_lr__random_state": [42],
    #"clf_lr__class_weight": ["balanced"],
    
    "clf_lr__solver": ["sag"],
    "clf_lr__penalty": ["l2"],
    "clf_lr__max_iter": [2000],
    "clf_lr__n_jobs":[-1],
    "clf_lr__C":[1.5],

    # TFIDF
    
    #"tfidf__stop_words":["english"],
    #"tfidf__sublinear_tf":[True],
    #"tfidf__preprocessor":[clean_text],
    
    "tfidf__analyzer": ["word"], 
    "tfidf__max_features":[120000],
    "tfidf__max_df": [0.7],
    "tfidf__min_df": [2],
    "tfidf__ngram_range": [(1,2)]
}]

lr_gridcv = GridSearchCV(estimator=lr_pipe, param_grid=lr_params, scoring=scoring, cv=skcv,  n_jobs=-1, verbose=2, refit= True)

lr_gridcv.fit(X_train_val, y_train_val)

lr_best_params = lr_gridcv.best_params_

print(lr_best_params)
print(lr_gridcv.best_score_)

"""
Tried Parameters with GridSearchCv

# Estimator

"clf_lr__solver": ["sag"], "clf_lr__penalty": ["l2"],
"clf_lr__solver": ["saga"], "clf_lr__penalty": ["l1", "l2", "elasticnet"],

"clf_lr__C":[0.1, 1, 1.5, 2],

# TFIDF

"tfidf__max_features":[50000, 80000, 100000],
"tfidf__max_df": [0.7, 0.75, 0.85, 0.9, 0.95],
"tfidf__min_df": [1, 2, 3],

"tfidf__analyzer": ["word"], "tfidf__ngram_range": [(1,1), (1, 2), (1, 3)]
"tfidf__analyzer": ["char_wb"], "tfidf__ngram_range": [(3, 5), (4, 6)]

"""
# Estimated Run time: 10 minutes 22 seconds

Fitting 5 folds for each of 1 candidates, totalling 5 fits
{'clf_lr__C': 1.5, 'clf_lr__max_iter': 2000, 'clf_lr__n_jobs': -1, 'clf_lr__penalty': 'l2', 'clf_lr__solver': 'sag', 'tfidf__analyzer': 'word', 'tfidf__max_df': 0.7, 'tfidf__max_features': 120000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}
0.7124183373840819


'\nTried Parameters with GridSearchCv\n\n# Estimator\n\n"clf_lr__solver": ["sag"], "clf_lr__penalty": ["l2"],\n"clf_lr__solver": ["saga"], "clf_lr__penalty": ["l1", "l2", "elasticnet"],\n\n"clf_lr__C":[0.1, 1, 1.5, 2],\n\n# TFIDF\n\n"tfidf__max_features":[50000, 80000, 100000],\n"tfidf__max_df": [0.7, 0.75, 0.85, 0.9, 0.95],\n"tfidf__min_df": [1, 2, 3],\n\n"tfidf__analyzer": ["word"], "tfidf__ngram_range": [(1,1), (1, 2), (1, 3)]\n"tfidf__analyzer": ["char_wb"], "tfidf__ngram_range": [(3, 5), (4, 6)]\n\n'

In [214]:
# Logistic Regression label base analyis
y_pred_lr = cross_val_predict(
    estimator = lr_gridcv.best_estimator_,
    X = X_train_val,
    y = y_train_val,
    cv=skcv
)

print(classification_report(y_train_val, y_pred_lr,labels=labels,target_names=targets))


              precision    recall  f1-score   support

      label6       0.79      0.69      0.74     23542
      label2       0.74      0.82      0.78     10588
      label3       0.82      0.83      0.82     11161
      label4       0.58      0.56      0.57      9977
      label1       0.80      0.93      0.86      8574
      label0       0.56      0.52      0.54     13053
      label5       0.58      0.82      0.68      3102

    accuracy                           0.72     79997
   macro avg       0.69      0.74      0.71     79997
weighted avg       0.72      0.72      0.71     79997



In [215]:
# SGDClassifier

sgd_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", sublinear_tf=True, preprocessor=clean_text)),
    ("clf_sgd", SGDClassifier(random_state=42, class_weight="balanced"))
])
sgd_params = [{
    
    # Estimator
    
    
    #"clf_sgd__random_state":[42], 
    #"clf_sgd__class_weight":["balanced"], 
    
    "clf_sgd__max_iter": [1000], 
    "clf_sgd__alpha": [0.001],
    "clf_sgd__loss": ["squared_hinge"],  
    "clf_sgd__n_jobs":[-1],

    # TFIDF
    
    #"tfidf__stop_words":["english"],
    #"tfidf__sublinear_tf":[True],
    #"tfidf__preprocessor":[clean_text],
    
    "tfidf__max_features":[100000],
    "tfidf__max_df": [0.7],
    "tfidf__min_df": [5],
    "tfidf__analyzer": ["word"],
    "tfidf__ngram_range": [(1,3)]   
}]


sgd_gridcv =  GridSearchCV(estimator=sgd_pipe, param_grid=sgd_params, scoring=scoring, cv=skcv,  n_jobs=-1, verbose=2, refit= True)

sgd_gridcv.fit(X_train_val, y_train_val)

sgd_best_params = sgd_gridcv.best_params_

print(sgd_best_params)
print(sgd_gridcv.best_score_)


"""
Tried Parameters with GridSearchCV

# Estimator

"clf_sgd__alpha": [0.001, 0.01, 0.1],
"clf_sgd__loss": ["hinge", "squared_hinge"],  


# TFIDF

"tfidf__max_features":[50000, 80000, 100000, 120000],
"tfidf__max_df": [0.7, 0.8, 0.9, 0.95],
"tfidf__min_df": [1, 2, 3, 5],

"tfidf__analyzer": ["word"], "tfidf__ngram_range": [(1,1), (1, 2), (1, 3)]
"tfidf__analyzer": ["char_wb"], "tfidf__ngram_range": [(3, 5), (4, 6)]

"""
# Estimated Run time: 2 minutes 19 seconds

Fitting 5 folds for each of 1 candidates, totalling 5 fits
{'clf_sgd__alpha': 0.001, 'clf_sgd__loss': 'squared_hinge', 'clf_sgd__max_iter': 1000, 'clf_sgd__n_jobs': -1, 'tfidf__analyzer': 'word', 'tfidf__max_df': 0.7, 'tfidf__max_features': 100000, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 3)}
0.691958397158352


'\nTried Parameters with GridSearchCV\n\n# Estimator\n\n"clf_sgd__alpha": [0.001, 0.01, 0.1],\n"clf_sgd__loss": ["hinge", "squared_hinge"],  \n\n\n# TFIDF\n\n"tfidf__max_features":[50000, 80000, 100000, 120000],\n"tfidf__max_df": [0.7, 0.8, 0.9, 0.95],\n"tfidf__min_df": [1, 2, 3, 5],\n\n"tfidf__analyzer": ["word"], "tfidf__ngram_range": [(1,1), (1, 2), (1, 3)]\n"tfidf__analyzer": ["char_wb"], "tfidf__ngram_range": [(3, 5), (4, 6)]\n\n'

In [216]:
# SGDClassifier label base analyis

y_pred_sgd = cross_val_predict(
    estimator = sgd_gridcv.best_estimator_,
    X = X_train_val,
    y = y_train_val,
    cv=skcv
)

print(classification_report(y_train_val, y_pred_sgd,labels=labels,target_names=targets))

              precision    recall  f1-score   support

      label6       0.73      0.77      0.75     23542
      label2       0.72      0.79      0.75     10588
      label3       0.78      0.81      0.80     11161
      label4       0.65      0.49      0.56      9977
      label1       0.74      0.96      0.83      8574
      label0       0.67      0.41      0.51     13053
      label5       0.51      0.87      0.65      3102

    accuracy                           0.71     79997
   macro avg       0.69      0.73      0.69     79997
weighted avg       0.71      0.71      0.70     79997



In [217]:
# MultinomialNB

nb_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", sublinear_tf=True, preprocessor=clean_text)),
    ("clf_nb", MultinomialNB())
])
nb_params = [{
    
    # Estimator
    
    "clf_nb__alpha": [0.1],
    
    
    # TFIDF

    #"tfidf__stop_words":["english"],
    #"tfidf__sublinear_tf":[True],
    #"tfidf__preprocessor":[clean_text],
    
    "tfidf__max_features":[100000],
    "tfidf__max_df": [0.7],
    "tfidf__min_df": [4],   
    "tfidf__analyzer": ["word"], 
    "tfidf__ngram_range": [(1,3)]
}]


nb_gridcv = GridSearchCV(estimator=nb_pipe, param_grid=nb_params, scoring=scoring, cv=skcv, n_jobs=-1, verbose=2, refit= True)

nb_gridcv.fit(X_train_val, y_train_val)

nb_best_params = nb_gridcv.best_params_

print(nb_best_params)
print(nb_gridcv.best_score_)


"""
# Estimator

"clf_nb__alpha": [0.01, 0.1, 0.5, 1],


# TFIDF

"tfidf__max_features":[50000, 80000, 100000, 120000],
"tfidf__max_df": [0.7, 0.8, 0.9, 0.95],
"tfidf__min_df": [1, 2, 3, 5],

"tfidf__analyzer": ["word"], "tfidf__ngram_range": [(1,1), (1, 2), (1, 3)]
"tfidf__analyzer": ["char_wb"], "tfidf__ngram_range": [(3, 5), (4, 6)]

"""
# Estimated Run time: 2 minutes 16 seconds

Fitting 5 folds for each of 1 candidates, totalling 5 fits
{'clf_nb__alpha': 0.1, 'tfidf__analyzer': 'word', 'tfidf__max_df': 0.7, 'tfidf__max_features': 100000, 'tfidf__min_df': 4, 'tfidf__ngram_range': (1, 3)}
0.6828230453295034


'\n# Estimator\n\n"clf_nb__alpha": [0.01, 0.1, 0.5, 1],\n\n\n# TFIDF\n\n"tfidf__max_features":[50000, 80000, 100000, 120000],\n"tfidf__max_df": [0.7, 0.8, 0.9, 0.95],\n"tfidf__min_df": [1, 2, 3, 5],\n\n"tfidf__analyzer": ["word"], "tfidf__ngram_range": [(1,1), (1, 2), (1, 3)]\n"tfidf__analyzer": ["char_wb"], "tfidf__ngram_range": [(3, 5), (4, 6)]\n\n'

In [218]:
# MultinomialNB label base analysis

y_pred_nb = cross_val_predict(
    estimator = nb_gridcv.best_estimator_,
    X = X_train_val,
    y = y_train_val,
    cv=skcv
)

print(classification_report(y_train_val, y_pred_nb, labels=labels,target_names=targets))

              precision    recall  f1-score   support

      label6       0.69      0.76      0.72     23542
      label2       0.71      0.77      0.74     10588
      label3       0.75      0.80      0.78     11161
      label4       0.65      0.52      0.57      9977
      label1       0.78      0.93      0.85      8574
      label0       0.59      0.40      0.48     13053
      label5       0.59      0.71      0.64      3102

    accuracy                           0.69     79997
   macro avg       0.68      0.70      0.68     79997
weighted avg       0.69      0.69      0.68     79997



# Comparing Models

In [219]:
from sklearn.model_selection import cross_val_score

models = {
    "LinearSVC": lsvc_gridcv.best_estimator_,
    "Logistic Regression": lr_gridcv.best_estimator_,
    "SGDClassifier": sgd_gridcv.best_estimator_,
    "MultinomialNB": nb_gridcv.best_estimator_
}

for name, pipe in models.items():
    score = cross_val_score(
        estimator=pipe, X=X_train_val, y=y_train_val, scoring="f1_macro", cv=skcv, n_jobs=-1
    )
    print(name, ": ", score.mean())
    
# Estimated Run time: 3 minutes 13 seconds

LinearSVC :  0.7168774285019877
Logistic Regression :  0.7124183373840819
SGDClassifier :  0.691958397158352
MultinomialNB :  0.6828230453295034


-   As a result LinearSVC is the best and fastest model between these four model.
-   That's why I decided to use LinearSVC
